# Setup

In [1]:
import numpy as np, timeit, time, matplotlib.pyplot as plt, json, os
from tqdm import tqdm
from collections import Counter

In [4]:
from binary_models import *
from actualcauses_local.base_algorithm import beam_search
from actualcauses_local.iterative_subinstance_algorithm import iterative_identification
from actualcauses_local.scm_class import SCM

from general import *

# Functions

In [5]:
def render_instance(datum, instance):
    variables = get_SMK_dim_labels(datum["n_attacker"])
    for i in range(6*datum["n_attacker"]):
        print(variables[i], instance["actual_state"][i], end = " / ")
    print()
    for j in range(i, i+3*datum["n_attacker"]):
        print(variables[j], instance["actual_state"][j], end = " / ")
    print()
    for k in range(j, j+2*datum["n_attacker"]):
        print(variables[k], instance["actual_state"][k], end = " / ")
    print()
    print("DK", instance["actual_state"][-3], end = " / ")
    print("SD", instance["actual_state"][-2], end = " / ")
    print("DK", instance["actual_state"][-1])

In [6]:
def show_variables(cause, variables):
    print(list(map(lambda x: variables[x], cause)))

In [7]:
def show_causes(causes, variables):
    for cause in causes:
        show_variables(cause, variables)

# Main

In [10]:
def run_pb(u, e):
    rules = dict(e)
    s = [None] * 6
    x, y = u
    X,Y,A,B,C,T = range(6)
    s[X] = rules.get(X, x)
    s[Y] = rules.get(Y, y)
    s[A] = rules.get(A, s[X])
    s[B] = rules.get(B, s[X] and s[Y])
    s[C] = rules.get(C, s[Y] and not s[X])
    
    s[T] = s[A] or s[B] or s[C]
    return s

heuristic = lambda v: sum(v)-1

scm = SCM(
    V="X Y A B C T".split(" "),
    U="x y".split(" "),
    D=[(0,1)] * 6,
    F=run_pb,
    u=(1,1),
    psi=heuristic,
    dag=[[],[],[0],[0,1],[0,1],[2,3,4]]
)

In [11]:
out_base = beam_search(**scm.get_input_beam_search(),beam_size=-1)
out_ISI = iterative_identification(**scm.get_input_ISI(), beam_size=-1)

In [12]:
print("Base algo:")
show_rules(out_base, scm.V)
print()
print("ISI algo")
show_rules(out_ISI, scm.V)

Base algo:
C={'X': '0'}, W={'C': '0'}, output=0, score=0.000
C={'Y': '0', 'A': '0'}, W={}, output=0, score=0.000
C={'A': '0', 'B': '0'}, W={}, output=False, score=1.000

ISI algo
C={'A': '0', 'B': '0'}, W={}, output=False, score=1.000
C={'X': '0', 'Y': '0'}, W={}, output=0, score=-1.000
C={'Y': '0', 'A': '0'}, W={}, output=0, score=0.000


In [ ]:
"A".lower()